# 🔄 Evaluator-Optimizer 패턴

## 개념 소개
**Evaluator-Optimizer** 패턴은 두 개의 LLM이 협력하는 구조입니다:
- **Generator (생성자)**: 결과물을 생성합니다 (예: 광고 문구 작성)
- **Evaluator (평가자)**: 결과물을 기준에 따라 평가하고 피드백을 제공합니다

평가 결과가 기준을 통과하지 못하면, 피드백을 바탕으로 Generator가 다시 생성을 시도합니다.  
이 과정을 반복(Loop)하여 점점 더 나은 결과물을 만들어냅니다.

```
[START] → [Generator] → [Evaluator] → PASS → [END]
                  ↑____________| FAIL (피드백 반영 후 재시도)
```

In [ ]:
# ============================================================
# 📦 필수 라이브러리 임포트
# ============================================================
# LangGraph의 핵심 구성요소들을 불러옵니다
from langgraph.graph import StateGraph, START, END

# LangChain의 모델 초기화 함수
from langchain.chat_models import init_chat_model

# 타입 힌트를 위한 라이브러리
from typing import TypedDict, Literal

# Pydantic: 구조화된 출력(Structured Output)을 정의할 때 사용
from pydantic import BaseModel, Field

## ① Model 정의

In [ ]:
# ============================================================
# 🤖 LLM 모델 초기화
# ============================================================
# temperature=0.7: 창의적인 광고 문구 생성을 위해 약간의 무작위성을 허용
# temperature가 높을수록 → 더 창의적, 낮을수록 → 더 일관되고 예측 가능
model = init_chat_model("gpt-4o-mini", temperature=0.7)

## ② State 정의

**State**는 그래프 전체를 흐르는 **공유 데이터 저장소**입니다.  
모든 노드는 이 State를 읽고, 자신의 작업 결과를 State에 기록합니다.

In [26]:
# ============================================================
# 📋 AdState: 광고 문구 생성 워크플로우의 상태 정의
# ============================================================
# TypedDict를 사용하면 딕셔너리의 키와 값 타입을 명시적으로 정의할 수 있습니다.
# LangGraph의 StateGraph는 이 구조를 기반으로 상태를 관리합니다.
class AdState(TypedDict):
    product_name: str          # 광고할 상품명 (입력값, 변경 없음)
    ad_copy: str               # Generator가 작성한 광고 문구
    feedback: str              # Evaluator가 제공하는 피드백 (수정 지시사항)
    status: str                # 평가 결과: "pass" (합격) / "fail" (불합격)
    iteration_count: int       # 현재까지 시도한 횟수 (무한 루프 방지용)

In [27]:
# ============================================================
# 📊 EvaluationResult: Evaluator LLM의 출력 스키마 정의
# ============================================================
# BaseModel을 상속하면 Pydantic의 데이터 검증 기능을 활용할 수 있습니다.
# with_structured_output()과 함께 사용하면 LLM이 이 형식으로만 응답하도록 강제할 수 있습니다.
class EvaluationResult(BaseModel):
    # Literal 타입: "pass" 또는 "fail" 두 값 중 하나만 허용
    status: Literal["pass", "fail"] = Field(description="기준 충족 여부")
    
    # 불합격(fail) 시 어떻게 수정해야 하는지 구체적인 지시사항
    feedback: str = Field(description="탈락 시 구체적인 수정 지시사항")

In [28]:
# ============================================================
# 🔗 Evaluator 전용 LLM 생성
# ============================================================
# with_structured_output(EvaluationResult):
#   → LLM의 응답을 EvaluationResult 객체로 자동 파싱해줍니다.
#   → 일반 텍스트 응답이 아닌, 정해진 형식(status, feedback)으로만 답변을 받습니다.
#   → 내부적으로 Function Calling 또는 JSON Mode를 활용합니다.
evaluator_llm = model.with_structured_output(EvaluationResult)

## ③ Node 정의

각 노드는 State를 입력받아 처리 후 **변경된 State의 일부**를 딕셔너리로 반환합니다.  
반환하지 않은 키는 이전 값이 그대로 유지됩니다.

In [32]:
# ============================================================
# ✍️ [Generator 노드] copywriter_node - 광고 문구 생성 담당
# ============================================================
# 역할: 신입 카피라이터처럼 광고 문구를 작성합니다.
#   - 첫 번째 시도: 의도적으로 건조하게 작성 (Evaluator의 피드백 유발용)
#   - 두 번째 이후: 팀장(Evaluator)의 피드백을 반영하여 개선된 문구 작성
def copywriter_node(state: AdState):
    product = state["product_name"]       # 상품명 읽기
    feedback = state.get("feedback")      # 이전 피드백 읽기 (없으면 None)
    count = state.get("iteration_count", 0)  # 현재 시도 횟수 (기본값 0)
        
    print(f"\n--- [Copywriter] 광고 문구 작성 중 (시도: {count + 1}) ---")

    if not feedback:
        # ① 첫 번째 시도: 일부러 건조하게 작성하여 Evaluator가 피드백을 주도록 유도
        #    → 이렇게 설계하면 워크플로우의 "루프" 동작을 확인할 수 있습니다
        prompt = f"'{product}'의 기능 위주로 인스타그램 홍보 문구를 건조하게 작성해줘. 홍보 문구만 답변하고 반드시 20자 이하로 작성하시오."
    else:
        # ② 피드백이 있으면: 수정 지시사항을 명확히 전달하여 개선된 문구 요청
        #    XML 태그(<>)를 사용하면 LLM이 지시사항을 더 명확히 인식합니다
        prompt = f"""
        '{product}' 인스타그램 홍보 문구를 다시 작성해.

        <반드시 지켜야 할 수정 사항>
        {feedback}
        </반드시 지켜야 할 수정 사항>

        <작성 시 반드시 지켜야할 사항>
        홍보 문구만 답변하고 절대 50자 이하로 작성하시오.
        </작성 시 반드시 지켜야할 사항>
        """
    
    msg = model.invoke(prompt)
    
    # 변경된 State 항목만 반환: 새 광고 문구 + 증가한 시도 횟수
    return {"ad_copy": msg.content, "iteration_count": count + 1}

In [33]:
# ============================================================
# 🧑‍💼 [Evaluator 노드] manager_node - 광고 문구 평가 담당
# ============================================================
# 역할: 깐깐한 마케팅 팀장처럼 광고 문구를 심사합니다.
#   - 정량 기준 + 정성 기준을 모두 적용
#   - 불합격 시 구체적인 수정 지시사항(feedback)을 함께 반환
def manager_node(state: AdState):
    ad_copy = state['ad_copy']  # 신입이 작성한 광고 문구 읽기
    print(f"\n--- [Manager] 문구 검수 중 ---")
    print(f"   ㄴ 신입이 쓴 글: {ad_copy}")

    # 평가 프롬프트: 구체적인 평가 기준을 명시할수록 일관된 판단을 얻을 수 있습니다
    prompt = f"""
    당신은 깐깐한 마케팅 팀장입니다. 신입 사원이 쓴 다음 광고 문구를 평가하세요:

    "{ad_copy}"

    <평가 기준>
    1. (정량) 해시태그(#)가 3개 이상 있어야 합니다.
    2. (정량) '할인' 또는 '특가'라는 단어가 포함되어야 합니다.
    3. (정성 - 중요!) **문구가 너무 설명문 같거나 딱딱하면 안 됩니다. 소비자의 감성을 자극하는 '활기차고 매력적인 톤'이어야 합니다.**
    </평가 기준>

    위 3가지 기준 중 하나라도 부족하면 fail을 주세요.
    특히 3번(톤앤매너)이 부족하다면 "좀 더 감성적으로 쓰세요" 같이 100자 이내로 조언하세요.
    """
    
    # evaluator_llm은 with_structured_output으로 생성되어 EvaluationResult 객체를 반환합니다
    result = evaluator_llm.invoke(prompt)

    print(f"    ㄴ 판정: {result.status.upper()}")
    if result.status == "fail":
        print(f"    ㄴ 지적: {result.feedback}")

    # 평가 결과(status)와 피드백(feedback)을 State에 저장
    return {"status": result.status, "feedback": result.feedback}

## ④ 그래프 생성

**조건부 엣지(Conditional Edge)**를 사용하여 루프를 구현합니다.  
평가 결과에 따라 다음 노드를 동적으로 결정합니다.

In [34]:
# ============================================================
# 🔀 라우팅 함수 - 루프 계속 여부 결정
# ============================================================
# 이 함수는 노드가 아니라 "조건부 엣지"에서 사용됩니다.
# 반환값: 다음으로 이동할 노드 이름 또는 END (종료)
def route_submission(state: AdState):
    status = state['status']                # 현재 평가 결과
    iteration_count = state['iteration_count']  # 현재까지 시도 횟수

    # 조건 ①: 합격한 경우 → 워크플로우 종료
    if status == 'pass':
        print('✅ PASS! 최종 광고 문구가 승인되었습니다.')
        return END
    
    # 조건 ②: 최대 시도 횟수(3회)를 초과한 경우 → 강제 종료 (무한 루프 방지)
    if iteration_count >= 3:
        print('⛔ 3번 수정했음에도 불합격... 작업을 종료합니다.')
        return END

    # 조건 ③: 아직 시도 횟수가 남아있고 불합격인 경우 → 다시 Generator로 이동
    return 'copywriter_node'

In [35]:
# ============================================================
# 🏗️ 그래프 조립 (StateGraph 구성)
# ============================================================

# 1단계: StateGraph 인스턴스 생성 (AdState 형태의 상태를 사용)
workflow = StateGraph(AdState)

# 2단계: 노드 등록 (이름, 함수)
workflow.add_node("copywriter_node", copywriter_node)  # Generator 노드
workflow.add_node("manager_node", manager_node)        # Evaluator 노드

# 3단계: 일반 엣지 추가 (항상 이동)
workflow.add_edge(START, 'copywriter_node')             # 시작 → 카피라이터
workflow.add_edge('copywriter_node', 'manager_node')   # 카피라이터 → 팀장

# 4단계: 조건부 엣지 추가 (조건에 따라 다른 경로로 이동)
# route_submission 함수의 반환값에 따라 'copywriter_node' 또는 END로 이동
workflow.add_conditional_edges(
    'manager_node',        # 이 노드의 출력을 평가
    route_submission,      # 경로를 결정하는 함수
    ['copywriter_node', END]  # 가능한 목적지 목록 (명시적으로 선언)
)

In [36]:
# ============================================================
# ⚙️ 그래프 컴파일
# ============================================================
# compile()을 호출하면 실행 가능한 앱 객체(app)가 생성됩니다.
# 이 단계에서 그래프의 구조가 검증되고 최적화됩니다.
app = workflow.compile()

In [ ]:
# ============================================================
# 🗺️ 그래프 구조 시각화
# ============================================================
# Jupyter 환경에서 app을 단독으로 실행하면 그래프 구조를 이미지로 확인할 수 있습니다.
# 노드와 엣지, 루프 구조를 한눈에 파악하는 데 유용합니다.
app

## ⑤ 실행

In [ ]:
# ============================================================
# 🚀 워크플로우 실행
# ============================================================
# invoke(): 동기 방식으로 그래프 전체를 실행하고 최종 State를 반환합니다.
# inputs: 초기 State 값 (product_name만 입력, 나머지는 None/기본값으로 시작)
inputs = {"product_name": "자율주행 자동차"}
result = app.invoke(inputs)

In [ ]:
# ============================================================
# 📋 최종 결과 확인
# ============================================================
# result는 최종 AdState 딕셔너리입니다.
# 최종 광고 문구, 마지막 피드백, 최종 상태, 총 시도 횟수를 확인할 수 있습니다.
result

---

# 🏢 Orchestrator-Worker 패턴

## 개념 소개
**Orchestrator-Worker** 패턴은 **병렬 처리**를 위한 구조입니다:
- **Orchestrator (지휘자)**: 큰 작업을 여러 개의 하위 작업으로 분배합니다
- **Worker (작업자)**: 각자 맡은 하위 작업을 독립적으로 수행합니다 (병렬 실행)
- **Synthesizer (통합자)**: 완료된 모든 작업 결과를 하나로 합칩니다

```
                    ┌→ [Worker 1] ─┐
[START] → [Orchestrator] → [Worker 2] → [Synthesizer] → [END]
                    └→ [Worker 3] ─┘
         (계획 수립)  (병렬 실행)         (결과 취합)
```

**핵심 기술**: LangGraph의 **`Send` API**를 사용하여 동적으로 Worker를 생성합니다.

## ① Model 정의

In [40]:
# ============================================================
# 🤖 LLM 모델 초기화
# ============================================================
# 보고서 작성은 일관된 내용이 중요하므로 temperature를 지정하지 않아 기본값(0)을 사용합니다.
model = init_chat_model("gpt-4o-mini")

## ② State 정의

### ⚡ 핵심 개념: `Annotated`와 `operator.add`
여러 Worker가 동시에 결과를 반환할 때, 각 결과를 **덮어쓰지 않고 누적**해야 합니다.  
`Annotated[List[str], operator.add]`를 사용하면 리스트에 자동으로 **append** 됩니다.

In [41]:
from typing import List

# ============================================================
# 📄 Section: 보고서 각 섹션(목차)의 데이터 구조
# ============================================================
# Orchestrator가 생성하는 계획표의 각 항목을 나타냅니다.
# Pydantic BaseModel을 사용하여 structured output으로 받을 수 있습니다.
class Section(BaseModel):
    name: str = Field(description="목차(섹션)의 제목")          # 예: "1장. AI의 역사"
    description: str = Field(description="이 섹션에서 다뤄야 할 핵심 내용")  # 집필 가이드

In [42]:
import operator
from typing import Annotated

# ============================================================
# 📋 ReportState: 보고서 작성 워크플로우의 전체 상태
# ============================================================
class ReportState(TypedDict):
    topic: str                                              # 보고서 주제 (입력값)
    sections: List[Section]                                 # Orchestrator가 계획한 목차 목록

    # ⚡ 핵심: Annotated + operator.add
    # 여러 Worker가 각자의 completed_sections를 반환할 때,
    # 덮어쓰기(overwrite)가 아닌 합치기(append)가 됩니다.
    # 예: Worker1이 ["섹션1"] 반환, Worker2가 ["섹션2"] 반환
    #     → State의 completed_sections = ["섹션1", "섹션2"]
    completed_sections: Annotated[List[str], operator.add]

    final_report: str  # Synthesizer가 완성한 최종 보고서

In [43]:
# ============================================================
# 📝 WorkerState: Worker 노드 전용 상태 (작업 지시서)
# ============================================================
# Worker는 전체 ReportState가 아닌, 자신이 담당하는 Section만 알면 됩니다.
# Send API를 통해 각 Worker에게 개별적으로 전달되는 "작업 지시서" 역할을 합니다.
class WorkerState(TypedDict):
    section: Section  # 이 Worker가 작성해야 할 단일 섹션 정보

In [44]:
# ============================================================
# 🗂️ Plan: Orchestrator LLM의 출력 스키마
# ============================================================
# Orchestrator가 structured output으로 계획표를 반환할 때 사용하는 형식입니다.
class Plan(BaseModel):
    sections: List[Section] = Field(description="보고서 작성을 위한 목차 리스트")

In [45]:
# ============================================================
# 🔗 Orchestrator(계획자) 전용 LLM 생성
# ============================================================
# planner_llm은 Plan 객체를 반환합니다.
# → 즉, LLM의 응답이 자동으로 {sections: [Section, Section, ...]} 형태로 파싱됩니다.
planner_llm = model.with_structured_output(Plan)

## ③ Node 정의

In [46]:
# ============================================================
# 🎯 [Orchestrator 노드] orchestrator_node - 보고서 목차 계획 수립
# ============================================================
# 역할: 주어진 주제에 대해 보고서 구조(목차)를 설계합니다.
#   - structured output을 활용하여 목차를 Section 객체 리스트로 받아옵니다.
#   - 이후 assign_workers 함수가 이 목차를 기반으로 Worker들에게 작업을 분배합니다.
def orchestrator_node(state: ReportState):
    topic = state["topic"]
    print(f"\n--- [Orchestrator] '{topic}' 보고서 계획 수립 중 ---")

    # 목차 생성 (최대 3개로 제한하여 실습 속도 조절)
    plan = planner_llm.invoke(f"'{topic}'에 대한 보고서 목차를 짜줘. 3개 섹션 이내로 구성해.")

    print(f"📋 생성된 계획: {[s.name for s in plan.sections]}")
    
    # sections를 State에 저장 → assign_workers가 이 값을 읽어서 Worker를 생성합니다
    return {"sections": plan.sections}

In [47]:
# ============================================================
# 👷 [Worker 노드] worker_node - 담당 섹션 집필
# ============================================================
# 역할: 자신에게 할당된 섹션 하나를 집필합니다.
#   - 여러 개의 Worker가 각기 다른 섹션을 동시에(병렬로) 처리합니다.
#   - 입력 State는 ReportState가 아닌 WorkerState (섹션 정보만 포함)
#   - 결과는 completed_sections 리스트에 누적됩니다 (operator.add 덕분에)
def worker_node(state: WorkerState):
    section = state["section"]  # 이 Worker에게 할당된 섹션 정보
    print(f"   --- [Worker] 집필 중: {section.name} ---")

    # 섹션 제목과 내용 가이드를 바탕으로 글쓰기 요청
    prompt = f"""
    다음 섹션에 대한 내용을 짧게 작성해줘.
    제목: {section.name}
    내용 가이드: {section.description}
    """
    msg = model.invoke(prompt)

    # 마크다운 형식으로 포맷팅 (## 헤딩 추가)
    content = f"## {section.name}\n{msg.content}\n"

    # ⚡ 리스트로 감싸서 반환해야 합니다!
    # Annotated[List[str], operator.add] 덕분에 여러 Worker의 결과가 자동으로 합쳐집니다.
    return {"completed_sections": [content]}

In [48]:
# ============================================================
# 📚 [Synthesizer 노드] synthesizer_node - 원고 취합 및 최종 편집 (v1: 단순 합치기)
# ============================================================
# 역할: 모든 Worker가 완료된 후, 각 섹션을 하나의 보고서로 합칩니다.
#   - 이 버전은 단순히 텍스트를 이어붙이는 방식입니다.
#   - 아래 "develop" 섹션에서 LLM을 활용한 개선 버전으로 업그레이드합니다.
def synthesizer_node(state: ReportState):
    print("\n--- [Synthesizer] 모든 원고 취합 및 최종 편집 ---")

    # completed_sections: 각 Worker가 작성한 섹션 텍스트 리스트
    completed = state["completed_sections"]
    
    # 단순히 줄바꿈으로 연결하여 최종 보고서 완성
    final_report = "\n".join(completed)

    return {"final_report": final_report}

## ④ 그래프 생성

### 🔑 Send API 소개
일반 `add_edge`는 **고정된 노드**로만 이동합니다.  
반면 **`Send` API**는 런타임에 **동적으로 여러 노드를 생성**하여 병렬 실행할 수 있게 합니다.

```python
# 일반 엣지: 항상 동일한 노드로 이동
workflow.add_edge("A", "B")

# Send API: 섹션 수만큼 worker_node를 동적으로 생성하여 병렬 실행
return [Send("worker_node", {"section": s}) for s in sections]
```

### [Send API 공식 문서](https://reference.langchain.com/python/langgraph/types/?h=send#langgraph.types.Send)

In [49]:
from langgraph.types import Send

# ============================================================
# 🌐 동적 라우팅 함수 - Map 단계 (1개의 계획 → N개의 작업)
# ============================================================
# 역할: Orchestrator가 만든 목차 리스트를 읽어, 섹션 수만큼 Worker를 동적으로 생성합니다.
#
# Send(노드이름, 전달할_상태) 구조:
#   - 첫 번째 인자: 실행할 노드 이름 ("worker_node")
#   - 두 번째 인자: 해당 Worker에게 전달할 State (WorkerState 형식)
#
# 예: sections = [섹션A, 섹션B, 섹션C] 이면,
#     Worker 3개가 동시에(병렬로) 실행됩니다.
def assign_workers(state: ReportState):
    sections = state["sections"]
    
    # 리스트 컴프리헨션으로 섹션 수만큼 Send 객체 생성
    # 각 Send는 독립적인 Worker 실행 요청을 나타냅니다
    return [Send("worker_node", {"section": s}) for s in sections]

In [ ]:
# ============================================================
# 🏗️ 그래프 조립 (Map-Reduce 패턴)
# ============================================================

# StateGraph 생성 (ReportState 형태의 상태 사용)
workflow = StateGraph(ReportState)

# 노드 등록
workflow.add_node("orchestrator_node", orchestrator_node)  # 계획 수립 (Orchestrator)
workflow.add_node("worker_node", worker_node)              # 섹션 집필 (Worker, 복수 실행 가능)
workflow.add_node("synthesizer_node", synthesizer_node)    # 결과 취합 (Synthesizer)

# 엣지 1: 시작 → Orchestrator (계획 수립)
workflow.add_edge(START, "orchestrator_node")

# 엣지 2: Orchestrator → (Map) → Worker들 (조건부 엣지 + Send API)
# assign_workers 함수가 섹션 수만큼 Send를 반환하여 병렬 Worker를 생성합니다.
workflow.add_conditional_edges(
    "orchestrator_node",
    assign_workers,
    ["worker_node"]  # 생략 가능하지만, 명시적으로 작성하면 가독성이 좋아집니다
)

# 엣지 3: 모든 Worker → Synthesizer (Reduce 단계)
# 모든 Worker가 완료되면 자동으로 synthesizer_node로 이동합니다.
workflow.add_edge("worker_node", "synthesizer_node")

# 엣지 4: Synthesizer → 종료
workflow.add_edge("synthesizer_node", END)

In [51]:
# 그래프 컴파일
app = workflow.compile()

In [ ]:
# 그래프 구조 시각화
# Orchestrator에서 여러 Worker로 갈라지는 병렬 구조를 확인할 수 있습니다.
app

## ⑤ 실행

In [ ]:
# ============================================================
# 🚀 워크플로우 실행 (v1: 단순 취합)
# ============================================================
inputs = {"topic": "생성형 AI의 미래"}
result = app.invoke(inputs)

In [ ]:
# ============================================================
# 📄 최종 보고서 출력
# ============================================================
# completed_sections의 텍스트들이 단순히 이어붙여진 결과를 확인합니다.
print(result["final_report"])

---

### 🚀 업그레이드: Synthesizer 개선 (v2: LLM을 활용한 "화학적 결합")

v1의 단순 이어붙이기 방식을 개선합니다.  
Worker들이 각자 작성한 초안을 LLM 편집자가 **하나의 자연스러운 보고서로 재구성**합니다.  
서론/결론도 자동으로 추가되어 완성도 높은 결과물을 생성합니다.

In [56]:
# ============================================================
# 📚 [Synthesizer 노드 v2] synthesizer_node - LLM 기반 전문 편집
# ============================================================
# 개선점: 단순 이어붙이기 → LLM이 전체 내용을 재구성
#   - 각 섹션의 연결이 매끄럽게 처리됩니다
#   - 서론과 결론이 자동으로 추가됩니다
#   - 마크다운 형식으로 가독성이 향상됩니다
def synthesizer_node(state: ReportState):
    topic = state["topic"]
    completed_docs = state["completed_sections"]

    print(f"\n--- [Synthesizer] 원고 {len(completed_docs)}건 도착. 최종 편집 시작 ---")

    # 1단계: 모든 Worker의 결과물을 텍스트 덩어리로 병합
    raw_content = "\n\n".join(completed_docs)

    # 2단계: LLM에게 전문 편집자 역할을 부여하여 완성도 높은 보고서 생성
    prompt = f"""
    당신은 전문 리포트 편집자입니다.
    다음은 '{topic}'에 대해 여러 작가가 나누어 쓴 원고들입니다.

    이 초안들을 바탕으로 **하나의 자연스럽고 전문적인 보고서**로 다시 작성해주세요.

    [지시사항]
    1. 각 섹션의 연결이 매끄러워야 합니다.
    2. 전체를 아우르는 '서론'과 '결론'을 추가해주세요.
    3. 마크다운(Markdown) 형식을 사용하여 가독성을 높여주세요.

    [원고 내용]
    {raw_content}
    """

    # LLM이 전체 보고서를 재작성
    msg = model.invoke(prompt)
    return {'final_report': msg.content}

In [57]:
# ============================================================
# 🏗️ 그래프 재조립 (개선된 synthesizer_node 적용)
# ============================================================
# 노드 구조와 엣지는 동일하지만, synthesizer_node 함수만 v2로 교체됩니다.
# 파이썬에서 함수 재정의 시, 동일한 이름의 새 함수가 기존 함수를 덮어씁니다.
workflow = StateGraph(ReportState)

workflow.add_node("orchestrator_node", orchestrator_node)
workflow.add_node("worker_node", worker_node)
workflow.add_node("synthesizer_node", synthesizer_node)  # ← v2 버전이 사용됩니다

# 시작 → 팀장
workflow.add_edge(START, "orchestrator_node")

# 팀장 → (Map) → 팀원들 (Send API 활용)
workflow.add_conditional_edges(
    "orchestrator_node",
    assign_workers,
    ["worker_node"]
)

# 팀원들 → (Reduce) → 편집자
workflow.add_edge("worker_node", "synthesizer_node")

# 편집자 → 종료
workflow.add_edge("synthesizer_node", END)

In [58]:
# 개선된 그래프 컴파일
app = workflow.compile()

In [ ]:
# 그래프 구조 시각화 (v2)
app

In [ ]:
# ============================================================
# 🚀 워크플로우 실행 (v2: LLM 편집 버전)
# ============================================================
inputs = {"topic": "생성형 AI의 미래"}
result = app.invoke(inputs)

In [ ]:
# ============================================================
# 📄 최종 보고서 출력 (v2)
# ============================================================
# v1과 비교했을 때, 서론/결론이 추가되고 섹션 간 연결이 매끄러워진 것을 확인할 수 있습니다.
print(result["final_report"])

---

# 🤖 Agents (에이전트)

## 개념 소개
**Agent(에이전트)**는 LLM이 **스스로 판단**하여 도구(Tool)를 선택하고 실행하는 패턴입니다.  
단순히 텍스트를 생성하는 것을 넘어, **외부 함수를 직접 호출**할 수 있습니다.

### 동작 원리 (ReAct 패턴)
```
[START] → [LLM 판단] → 도구 호출 필요? → YES → [도구 실행] → [LLM 판단] (반복)
                                         ↓ NO
                                        [END]
```

1. LLM이 사용자 요청을 분석합니다
2. 도구 호출이 필요하면 → 도구를 실행하고 결과를 받습니다
3. 도구 호출이 불필요하면 → 최종 답변을 생성하고 종료합니다
4. 이 과정을 반복하며 복잡한 작업을 단계적으로 처리합니다

In [63]:
from langchain.tools import tool

# ============================================================
# 🔧 Tool 정의 - LLM이 사용할 수 있는 도구들
# ============================================================
# @tool 데코레이터를 사용하면 일반 파이썬 함수를 LangChain Tool로 변환합니다.
# 중요: docstring이 Tool의 설명(description)이 됩니다.
#       LLM은 이 설명을 읽고 어떤 상황에서 이 도구를 사용할지 판단합니다.

@tool
def multiply(a: int, b: int) -> int:
    """Multiply `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a * b  # a × b 결과 반환


@tool
def add(a: int, b: int) -> int:
    """Adds `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a + b  # a + b 결과 반환


@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a / b  # a ÷ b 결과 반환


# ============================================================
# 🔗 LLM에 도구 바인딩
# ============================================================
# bind_tools(): LLM에게 "이런 도구들을 사용할 수 있다"고 알려줍니다.
# → LLM이 필요에 따라 도구 호출(tool_calls)을 응답에 포함시킬 수 있게 됩니다.
tools = [add, multiply, divide]
tools_by_name = {tool.name: tool for tool in tools}  # 빠른 조회를 위한 딕셔너리
llm_with_tools = model.bind_tools(tools)              # 도구가 바인딩된 LLM

In [65]:
from langgraph.graph import MessagesState
from langchain.messages import SystemMessage, HumanMessage, ToolMessage


# ============================================================
# 🧠 [LLM 판단 노드] llm_call - 도구 호출 여부 결정
# ============================================================
# MessagesState: messages 키를 가진 특수 State로, 메시지 히스토리를 자동 관리합니다.
# 역할: 현재까지의 대화 내용을 바탕으로 LLM이 다음 행동을 결정합니다.
#   - 도구가 필요하다고 판단 → tool_calls가 포함된 응답 반환
#   - 충분히 답변 가능 → 일반 텍스트 응답 반환
def llm_call(state: MessagesState):
    """LLM decides whether to call a tool or not"""

    return {
        "messages": [
            llm_with_tools.invoke(
                [
                    # 시스템 메시지: LLM의 역할과 목적을 정의합니다
                    SystemMessage(
                        content="You are a helpful assistant tasked with performing arithmetic on a set of inputs."
                    )
                ]
                + state["messages"]  # 기존 대화 히스토리 추가
            )
        ]
    }


# ============================================================
# ⚙️ [도구 실행 노드] tool_node - 실제 도구 함수 호출
# ============================================================
# 역할: LLM이 요청한 도구 호출(tool_calls)을 실제로 실행합니다.
#   - LLM의 마지막 응답에서 tool_calls를 추출합니다
#   - 각 도구를 실행하고 결과를 ToolMessage로 반환합니다
#   - 반환된 ToolMessage는 다음 llm_call에서 "도구 실행 결과"로 참조됩니다
def tool_node(state: dict):
    """Performs the tool call"""

    result = []
    # LLM의 마지막 메시지에 포함된 모든 도구 호출 요청을 처리
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]      # 도구 이름으로 함수 조회
        observation = tool.invoke(tool_call["args"])  # 도구 실행 (실제 계산 수행)
        # ToolMessage: 도구 실행 결과를 LLM에게 전달하는 특수 메시지 타입
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}

In [66]:
# ============================================================
# 🔀 라우팅 함수 - 루프 계속 여부 결정
# ============================================================
# 역할: LLM의 응답을 확인하여 도구 노드로 이동할지, 종료할지 결정합니다.
# tool_calls가 있다 → 도구 실행 필요 → "tool_node"로 이동
# tool_calls가 없다 → 최종 답변 완성 → END (종료)
def should_continue(state: MessagesState) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""

    messages = state["messages"]
    last_message = messages[-1]  # 가장 최근 LLM 응답

    # LLM이 도구 호출을 요청한 경우 → tool_node로 라우팅
    if last_message.tool_calls:
        return "tool_node"

    # 도구 호출 없음 → 최종 답변으로 종료
    return END

In [ ]:
# ============================================================
# 🏗️ 에이전트 그래프 조립
# ============================================================

# MessagesState를 사용하는 StateGraph
agent_builder = StateGraph(MessagesState)

# 노드 등록
agent_builder.add_node("llm_call", llm_call)    # LLM 판단 노드
agent_builder.add_node("tool_node", tool_node)  # 도구 실행 노드

# 엣지 1: 시작 → LLM 판단
agent_builder.add_edge(START, "llm_call")

# 엣지 2: LLM 판단 → 도구 실행 또는 종료 (조건부)
# should_continue 함수가 "tool_node" 또는 END를 반환합니다
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    ["tool_node", END]  # 가능한 목적지
)

# 엣지 3: 도구 실행 → 다시 LLM 판단 (결과를 LLM에게 전달하여 재판단)
# 이 엣지가 루프를 만듭니다: LLM → 도구 → LLM → 도구 → ... → 종료
agent_builder.add_edge("tool_node", "llm_call")

# 그래프 컴파일
agent = agent_builder.compile()

# 그래프 구조 시각화
agent

In [ ]:
# ============================================================
# 🚀 에이전트 실행
# ============================================================
# 복합적인 요청: 덧셈 → 곱셈 → 이야기 생성
# LLM이 어떤 순서로 도구를 호출하는지 관찰해보세요!
messages = [HumanMessage(content="Add 3 and 4. Multiply the output by 7. And make a funny story about Japan.")]
messages = agent.invoke({"messages": messages})

# 전체 메시지 히스토리 출력
# HumanMessage → AIMessage(tool_calls) → ToolMessage → AIMessage(tool_calls) → ToolMessage → AIMessage(최종답변)
# 순서로 출력되는 것을 확인할 수 있습니다.
for m in messages["messages"]:
    m.pretty_print()

In [ ]:
# ============================================================
# 📋 전체 메시지 State 확인
# ============================================================
# 각 메시지의 타입과 내용을 상세히 확인합니다.
# tool_calls, tool_call_id 등의 속성을 통해 에이전트의 동작 흐름을 추적할 수 있습니다.
messages